# `torch.linalg.eig`

`torch.linalg.eig` takes a matrix input and decomposes it into eigenvalues and eigenvectors. The goal of this note is to understand how it can be implemented to run on a GPU. It is meant to be somewhat high to mid level, to give a fairly concise explanation which can be understood as long as you know some basic linear algebra and PyTorch. So I will purposefully leave out some details. To find these details, you can look at the links in the References section and links throughout the document. I heavily relied upon reference [1] when writing this, so that one is probably the most useful.

NOTE: Indices in the math formulas below start at 1, while indices in the code snippets start at 0. So don't get that confused.

## Recap of the eigenvalue problem

A square matrix $A$ describes a linear operator, which can be applied to a vector $u$ using matrix-vector multiplication to produce a new transformed vector $A u$.

In general $A u \ne a u$, where $a$ is a scalar. However, for every linear operator there is a set of nonzero vectors $v$, called eigenvectors, which do scale by some scalar value $a$, called an eigenvalue, when the operator is applied to them.

$$
A v = a v
\tag 1
$$

The goal of the eigenvalue problem is: given some operator $A$, find every eigenvalue $a$ that satisfies equation (1) and also find one eigenvector $v$ for each eigenvalue. An $n \times n$ matrix will have $n$ eigenvalues, though some may actually have the same value. The choice of eigenvectors is not unique, so it is often desirable to normalize them and also choose orthogonal eigenvectors if possible.

### Example: Characteristic equation

Let's say we're given the operator:

$$
A = \begin{pmatrix}
1 & 3 & 1 \\
0 & 2 & 0 \\
0 & 1 & 4
\end{pmatrix}
$$

One way to solve the eigenvalue problem by hand is to analytically solve what is called the characteristic equation or [characteristic polynomial](https://en.wikipedia.org/wiki/Characteristic_polynomial) of the matrix. This method is very limited because it only works for matrices of size $4 \times 4$ or smaller. An $n \times n$ matrix will produce an $n$-th degree polynomial, and only polynomials of degree 4 or less have analytic solutions in general. But it's easy to follow as an example, so it's worth taking a look at it. The characteristic equation is:

$$
\det(A - I a) = 0
$$

We just have to plug in $A$ and simplify to produce a set of equations to find the three eigenvalues.

$$
\det
\begin{pmatrix}
1 - a & 3 & 1 \\
0 & 2 - a & 0 \\
0 & 1 & 4 -a
\end{pmatrix}
= (1 - a) (2 - a) (4 - a) = 0
$$

The eigenvalues are:

$$
a_1 = 1 \quad
a_2 = 2 \quad
a_3 = 4
$$

Now we can find the eigenvectors individually.

$$
Av_1 = a_1 v_1
= \begin{pmatrix}
v_{11} + 3 v_{12} + v_{13} \\
2 v_{12} \\
v_{12} + 4 v_{13}
\end{pmatrix}
= \begin{pmatrix}
v_{11} \\ v_{12} \\ v_{13}
\end{pmatrix}
\rightarrow\quad
\begin{pmatrix}
3 v_{12} + v_{13} \\
v_{12} \\
v_{12} + 3 v_{13}
\end{pmatrix}
= 0
\quad\rightarrow\quad
\begin{matrix}
v_{11} = \text{arbitrary} \\
v_{12} = 0 \\
v_{13} = 0
\end{matrix}
$$

Since we want normalized eigenvectors, $v_{11} = 1$ is a good choice.

$$
v_1 = \begin{pmatrix}
1 \\ 0 \\ 0
\end{pmatrix}
$$

I won't write out the solution for the other two eigenvectors, because it's just more of the same. Let's see what `torch.linalg.eig` gives for this matrix.

In [1]:
import torch

r = torch.linalg.eig(torch.tensor([
    [1, 3, 1],
    [0, 2, 0],
    [0, 1, 4],
], dtype=torch.float))

# Note: `torch.linalg.eig` returns the individual eigenvectors as columns of a
# matrix, so we have to transpose.
for a, v in zip(r.eigenvalues, r.eigenvectors.T):
    print(f'eigenvalue: {a.item()}, eigenvector: {v}')

eigenvalue: (1+0j), eigenvector: tensor([1.+0.j, 0.+0.j, 0.+0.j])
eigenvalue: (4+0j), eigenvector: tensor([0.3162+0.j, 0.0000+0.j, 0.9487+0.j])
eigenvalue: (2+0j), eigenvector: tensor([ 0.9129+0.j,  0.3651+0.j, -0.1826+0.j])


All the eigenvalues match what we found above, and the eigenvector for $a_1 = 1$ also matches what we found.

## The QR algorithm

Since the characteristic equation method won't work in general, we need a more robust way to solve the eigenvalue problem. One of the most popular methods is called the [QR algorithm](https://en.wikipedia.org/wiki/QR_algorithm), and it is the method that `torch.linalg.eig` uses.

### QR iteration

The heart of the algorithm is called QR iteration. The first iteration starts with $A_1 = A$ and computes further iterations like so:

$$
Q_k, R_k = \text{decompQR}(A_k)
$$
$$
A_{k+1} = R_k Q_k
$$

$\text{decompQR}$ represents the [QR decomposition](https://en.wikipedia.org/wiki/QR_decomposition) of a matrix, and I won't go into how it is calculated. But it is worth noting that $A = QR$ by definition, and because of that:

$$
Q^{-1} A = R
\quad\rightarrow\quad
R Q = Q^{-1} A Q
$$

Because of some details I won't explain, $Q^{-1} A Q$ is a similarity transformation of $A$, which just means that it has the same eigenvalues as $A$.

After some number $n$ of iterations, the hope is that $A_n$ converges toward an upper triangular matrix, which means that all elements below the diagonal are (approximiately) zero. The diagonal elements of an upper triangular matrix are its eigenvalues, so since we're only applying similarity transformations, they will also be the eigenvalues of the original matrix $A$.

We'll check for convergence by taking the absolute value sums of each of the lower diagonal rows and compare them to zero. Once they're close enough to zero, we'll consider the matrix to be approximately triangular. This convergence check is based on the [Gershgorin circle theorem](https://en.wikipedia.org/wiki/Gershgorin_circle_theorem).

Let's try out QR iteration on the example matrix from earlier.

In [2]:
def is_approx_triangular(A, atol=1e-6, rtol=1e-6):
    lower_diag_sum = A.tril(diagonal=-1).abs().sum(dim=-1)
    return torch.allclose(
        lower_diag_sum,
        torch.zeros_like(lower_diag_sum),
        atol=atol,
        rtol=rtol
    )

# Returns (result, num iterations run, whether it converged)
def QR_iteration_naive(A, max_num_iters=1000):
    if not A.is_complex():
        # Convert to complex to allow for complex eigenvalues
        A = torch.complex(A, torch.zeros_like(A))

    for i in range(max_num_iters):
        Q, R = torch.linalg.qr(A)
        A = R @ Q

        if is_approx_triangular(A):
            return A, i + 1, True

    return A, max_num_iters, False

In [3]:
A = torch.tensor([
    [1, 3, 1],
    [0, 2, 0],
    [0, 1, 4],
], dtype=torch.float)

QR_iteration_naive(A)

(tensor([[ 1.0000e+00+0.j,  1.0000e+00+0.j, -3.0000e+00+0.j],
         [ 0.0000e+00+0.j,  4.0000e+00+0.j, -1.0000e+00+0.j],
         [ 0.0000e+00+0.j,  9.5368e-07+0.j,  2.0000e+00+0.j]]),
 22,
 True)

It took 22 iterations to converge on an approximately triangular matrix. And look, we've obtained the correct eigenvalues on the diagonal!

But convergence is not guaranteed for all matrices. Here's an example which will never converge:

In [4]:
A = torch.tensor([
    [0, -1],
    [1, 0],
], dtype=torch.float)

QR_iteration_naive(A)

(tensor([[ 0.+0.j, -1.+0.j],
         [ 1.+0.j,  0.+0.j]]),
 1000,
 False)

This matrix has complex eigenvalues $\pm i$, and basic QR iteration doesn't converge if there are two complex eigenvalues with the same norm. So we'll need to modify the algorithm in some way to account for that. Also, QR iteration like we did above is computationally expensive, and there is a cheaper way to obtain the same result by first converting the matrix into Hessenberg form.

### Hessenberg form

The idea is to apply a series of similarity transformations to matrix $A$ which turn it into a Hessenberg matrix. After that, we'll use a modified QR iteration algorithm that requires the input to be in Hessenberg form.

A Hessenberg matrix is almost upper triangular, except that any element directly below the diagonal can be non-zero. For instance, this is a Hessenberg matrix, where each $\times$ is an arbitrary value:

$$
\begin{pmatrix}
\times & \times & \times & \times & \times \\
\times & \times & \times & \times & \times \\
0 & \times & \times & \times & \times \\
0 & 0 & \times & \times & \times \\
0 & 0 & 0 &\times & \times
\end{pmatrix}
$$

To convert an $n \times n$ matrix to Hessenberg form, we'll apply $n-1$ transformations, indexed by $k \in [1, n-1]$. Each iteration operates on $A_k$, where $A_1 = A$, and produces an output matrix $A_{k + 1}$, which the next iteration will operate upon. In one of these iterations, we first take the vector $x$, which is defined as the last $n-k$ rows of the $k$-th column of $A_k$.

$$
x = A_{k(k+1:n, k)}
$$

And each iteration will endup zeroing out all but the first of those corresponding elements in the resulting $A_{k+1}$ matrix.

For instance, let's say we have a $5 \times 5$ matrix. We'll perform 4 iterations, where the $x$ vector is taken from the selected elements of $A_k$ below:

$$
\begin{matrix}
k = 1: \\
\\
\begin{pmatrix}
\times & \times & \times & \times & \times \\
\boxed{\times} & \times & \times & \times & \times \\
\boxed{\times} & \times & \times & \times & \times \\
\boxed{\times} & \times & \times & \times & \times \\
\boxed{\times} & \times & \times & \times & \times
\end{pmatrix}
\end{matrix}

\quad\rightarrow\quad

\begin{matrix}
k = 2: \\
\\
\begin{pmatrix}
\times & \times & \times & \times & \times \\
\times & \times & \times & \times & \times \\
0 & \boxed{\times} & \times & \times & \times \\
0 & \boxed{\times} & \times & \times & \times \\
0 & \boxed{\times} & \times & \times & \times
\end{pmatrix}
\end{matrix}

\quad\rightarrow\quad

\begin{matrix}
k = 3: \\
\\
\begin{pmatrix}
\times & \times & \times & \times & \times \\
\times & \times & \times & \times & \times \\
0 & \times & \times & \times & \times \\
0 & 0 & \boxed{\times} & \times & \times \\
0 & 0 & \boxed{\times} & \times & \times
\end{pmatrix}
\end{matrix}

\quad\rightarrow\quad

\text{etc.}
$$

In order to zero out these elements, given a particular $x$ vector, we first calculate the complex phase of $x_1$ (first element of $x$) using this formula:

$$
\rho = \begin{Bmatrix}
1 & \text{if } x_1 = 0 \\
-\exp\Bigl(i \text{ angle}(x_1)\Bigr) & \text{else}
\end{Bmatrix}
$$

Then we construct vector $u$ by starting with $x$, subtracting $\rho ||x||$ from its first element, where $||x||$ is the norm of $x$, and then normalizing the resulting vector.

In other words, where $e_1$ is the first basis vector $(1, 0, 0, ...)$ with $n-k$ elements, $u$ is:

$$
u = \frac{x - \rho ||x|| e_1}{|| x - \rho ||x|| e_1 ||}
$$

Then we construct a Householder reflector matrix $P$ from $u$. A Householder reflector is defined as any matrix of the form:

$$
P = I - 2 u u^*
$$

(Note: $uu^*$ is the outer product of $u$ and its complex conjugate, so it produces a matrix.)

But in our case, we actually need to we pad the first $k$ dimensions of $P$ like so, where $I_k$ is the identity matrix for $k$ dimensions:

$$
P = \begin{pmatrix}
I_{k} & 0 \\
0 & I_{n-k} - 2 u u^*
\end{pmatrix}
$$

Finally, we apply the Householder reflector matrix to $A_k$ to get the output for iteration $k$.

$$
A_{k+1} = P A_k P
$$

I'm not going to go into why this works, but below is a Python implementation of it, followed by a test that runs through a bunch of different matrices and shows that the calculated Hessenberg matrix is indeed Hessenberg and has the same set of eigenvalues as the original matrix.

One detail worth pointing out is that the `full_householder=False` setting enables a performance enhancement which avoids doing the two full $n \times n$ matrix multiplies in $P A_k P$, simplifying it to two matrix-vector products and two vector outer products. This is computationally cheaper, especially because the vectors get smaller with each iteration.

In [5]:
def hessenberg(A, full_householder=False):
    n = A.shape[0]
    A_k = A.clone()

    for k in range(n - 2):
        # Pick out the k-th column and all but the first k+1 rows.
        x = A_k[k + 1:, k]
        x_norm = torch.linalg.norm(x)

        # If x_norm is zero, then all the elements that this iteration aims to
        # zero out are already zeroed out, so we can skip it.
        if x_norm == 0:
            continue

        if x[0] == 0:
            rho = 1
        else:
            if x.is_complex():
                rho = -torch.exp(torch.angle(x[0]) * 1j)
            else:
                # Simplification for the real case
                rho = -torch.sign(x[0])

        u_k = x.clone()
        u_k[0] -= rho * x_norm

        u_k = u_k / torch.linalg.norm(u_k)

        if full_householder:
            P = torch.eye(n, dtype=u_k.dtype)
            P[k+1:, k+1:] -= 2 * u_k[:, None] @ u_k.conj()[None, :]
            A_k = P @ A_k @ P
        else:
            # vector-matrix product -> vector
            # vector outer product -> matrix
            A_k[k + 1:, k:] -= 2 * u_k[:, None] @ (u_k.conj() @ A_k[k+1:, k:])[None, :]
            # matrix-vector product -> vector
            # vector outer product -> matrix
            A_k[:, k+1:] -= 2 * (A_k[:, k+1:] @ u_k[:, None]) @ u_k.conj()[None, :]

    return A_k

In [6]:
# Compare two arrays of eigenvalues, allowing for them to be different orders.
def spectral_distance(wa, wb):
    D = torch.abs(wa[:, None] - wb[None, :])
    available = torch.ones_like(D, dtype=torch.bool)
    distances = []

    for _ in range(len(wa)):
        masked_D = torch.where(available, D, torch.inf)
        idx = torch.argmin(masked_D)
        i, j = torch.unravel_index(idx, D.shape)
        distances.append(D[i, j])
        available[i, :] = False
        available[:, j] = False

    return torch.stack(distances).max()

import itertools

cases = itertools.product(
    [torch.float, torch.complex64],
    range(1, 21),
    [False, True]
)

for dtype, n, full_householder in cases:
    for test_idx in range(20):
        A = torch.randn(n, n, dtype=dtype)
        H = hessenberg(A, full_householder)

        # Check that the output of `hessenberg` is actually a Hessenberg matrix
        for row in range(H.shape[0]):
            for col in range(H.shape[1]):
                if row > col + 1:
                    assert H[row, col].abs() < 1e-5

        # Check that the eigenvalues match
        A_eig = torch.linalg.eigvals(A)
        H_eig = torch.linalg.eigvals(H)
        dist = spectral_distance(A_eig, H_eig)
        assert dist < 1e-4

### Hessenberg QR iteration

Now that we know how to put any matrix into Hessenberg form, we can use a QR algorithm that is specifically for Hessenberg matrices.

The Hessenberg QR iteration is pretty similar to what we saw earlier, because it relies on repeatedly QR-decomposing a matrix and then doing the reverse matrix multiply, $RQ$. But there are two key differences.

The first difference is that before we decompose the matrix $H_k$ (at iteration number $k$), we perform a "shift" on it, and then undo the shift after the decomposition, like so:

$$
Q, R = \text{decompQR}(H_k - \sigma_k I)
$$
$$
H_{k+1} = RQ + \sigma_k I
$$

$\sigma_k$ is a scalar, and we'll talk about how to choose its value shortly.

The second difference is that each iteration of the Hessenberg QR algorithm aims to bring one particular target element of the matrix closer to 0, starting with the bottom right element under the diagonal. If that element is close enough to zero after some iteration, then the next iteration will move to the next element above and to the left of it.

Furthermore, the algorithm operates on an ever-shrinking submatrix. Once the target element has reached near zero, the active submatrix that the algorithm operates on will shrink by one row (from the bottom) and one column (from the right).

Below is a depiction of this progression for a $4 \times 4$ Hessenberg matrix. The box marks the $m \times m$ active submatrix, and the underlined element is the target element, which is in row $m-1$.

$$
\begin{matrix}
m=4 \\
\begin{pmatrix}
\boxed{\begin{matrix}
    \times & \times & \times & \times \\
    \times & \times & \times & \times \\
    0 & \times & \times & \times \\
    0 & 0 & \underline{\times} & \times
\end{matrix}}
\end{pmatrix}
\end{matrix}

\quad\rightarrow\quad

\begin{matrix}
m=3 \\
\begin{pmatrix}
\boxed{\begin{matrix}
    \times & \times & \times \\
    \times & \times & \times \\
    0 & \underline{\times} & \times
\end{matrix}}
& \begin{matrix}
\times \\
\times \\
\times
\end{matrix}
\\
\begin{matrix}
0 & 0 & 0 
\end{matrix}
& \times
\end{pmatrix}
\end{matrix}

\quad\rightarrow\quad

\begin{matrix}
m=2 \\
\begin{pmatrix}
\boxed{\begin{matrix}
    \times & \times \\
    \underline{\times} & \times
\end{matrix}}
&
\begin{matrix}
    \times & \times \\
    \times & \times
\end{matrix}
\\
\begin{matrix}
    0 & 0 \\
    0 & 0
\end{matrix}
&
\begin{matrix}
    \times & \times \\
    0 & \times
\end{matrix}
\end{pmatrix}
\end{matrix}

\quad\rightarrow\quad
\begin{pmatrix}
\times & \times & \times & \times \\
0 & \times & \times & \times \\
0 & 0 & \times & \times \\
0 & 0 & 0 & \times
\end{pmatrix}
$$

The final output is a diagonal matrix, and each iteration only applies similarity transformations, so the eigenvalues of the original matrix are found along the diagonal of the output.

The last detail is how to choose the shift values $\sigma_k$. One common choice is the Rayleigh quotient shift, which sets $\sigma_k$ to the value of the bottom right element of the $m \times m$ active submatrix of $H_k$, $\sigma_k = H_k[m,m]$. But that only works well for real eigenvalues. To account for complex eigenvalues, we'll use Wilkinson shift, which takes the bottom right $2 \times 2$ portion of the active submatrix, $H_k[m-1:m+1, m-1:m+1]$, finds its two eigenvalues, and chooses whichever one is closest to $H_k[m, m]$. The eigenvalues of a $2 \times 2$ matrix are simple to find, and we can derive the formula by hand by solving the characteristic equation.

$$
A = \begin{pmatrix}
a & b \\
c & d
\end{pmatrix}
$$

$$
\det(A - I\lambda) = 0
= \begin{pmatrix}
a - \lambda & b \\
c & d -\lambda
\end{pmatrix}
= (a - \lambda)(d - \lambda) - bc
$$

$$
= \lambda^2 - (a + d) \lambda + ad - bc = 0
$$

$$
\rightarrow\quad
\lambda = \frac{a + d \pm \sqrt{(a + d)^2 - 4 (ad - bc)}}{2}
$$

$$
\rightarrow\quad
\lambda = \frac{a + d \pm \sqrt{(a - d)^2 + 4bc}}{2}
$$

In [7]:
def wilkinson_shift(H, m):
    # Get the elements of the lower right 2x2 corner of the active submatrix
    a = H[m - 1, m - 1]
    b = H[m - 1, m]
    c = H[m, m - 1]
    d = H[m, m]

    left = (a + d) / 2
    right = (((a - d)**2 + 4 * b * c)**0.5) / 2

    eig1 = left + right
    eig2 = left - right

    # Pick the eigenvalue closest to d
    return eig1 if (eig1 - d).abs() < (eig2 - d).abs() else eig2

In [8]:
def QR_iteration_hessenberg_wilkinson(H, max_num_iters=1000):
    if not H.is_complex():
        # Convert to complex to allow for complex eigenvalues
        H = torch.complex(H, torch.zeros_like(H))
    else:
        H = H.clone()

    n = H.shape[-1]
    I = torch.eye(n, n, dtype=H.dtype)

    m = n - 1

    for i in range(max_num_iters):
        shift = wilkinson_shift(H, m)

        # Only iterate on the upper left (m+1)x(m+1) active submatrix
        I_shift = I[:m + 1, :m + 1] * shift
        Q, R = torch.linalg.qr(H[:m + 1, :m + 1] - I_shift)
        H[:m + 1, :m + 1] = R @ Q + I_shift

        # Apply the same similarity transform to the part of the rows above
        # the deflated block, so that H stays similar to the input
        #H[:m + 1, m + 1:] = Q.mH @ H[:m + 1, m + 1:]

        if H[m, m - 1].abs() < 1e-6:
            #H[m, m - 1] = 0
            m -= 1
            if m < 1:
                return H, i + 1, m, True

    return H, max_num_iters, m, False

In [9]:
A = torch.tensor([
    [1, 3, 1],
    [0, 2, 0],
    [0, 1, 4],
], dtype=torch.float)

QR_iteration_hessenberg_wilkinson(hessenberg(A))

(tensor([[ 1.0000+0.j, -2.2361+0.j,  2.2361+0.j],
         [ 0.0000+0.j,  2.0000+0.j,  1.0000+0.j],
         [ 0.0000+0.j,  0.0000+0.j,  4.0000+0.j]]),
 2,
 0,
 True)

In [10]:
A = torch.tensor([
    [0, -1],
    [1, 0],
], dtype=torch.float)

QR_iteration_hessenberg_wilkinson(hessenberg(A))

(tensor([[-4.2147e-08+1.0000e+00j, -8.9407e-08-5.9605e-08j],
         [ 2.1073e-08+8.8818e-16j, -1.9868e-08-1.0000e+00j]]),
 1,
 0,
 True)

In [11]:
cases = itertools.product(
    [torch.float, torch.complex64],
    range(1, 21),
)

for dtype, n in cases:
    for test_idx in range(20):
        A = torch.randn(n, n, dtype=dtype)
        H = hessenberg(A)
        eigvals = QR_iteration_hessenberg_wilkinson(H)[0].diagonal()

        # Check that the eigenvalues match
        eigvals_check = torch.linalg.eigvals(A)
        dist = spectral_distance(eigvals, eigvals_check)
        match = dist < 1e-3
        if not match:
            print(eigvals)
            print(eigvals_check)
        assert match

## References

[1] Lecture notes from [Numerical Methods for Solving Large Scale Eigenvalue Problems
(Spring semester 2018)](https://people.inf.ethz.ch/~arbenz/ewp/) by Prof. Dr. Peter Arbenz. Specifically [lecture 4 on the QR algorithm](https://people.inf.ethz.ch/arbenz/ewp/Lnotes/chapter4.pdf)

[2] <https://en.wikipedia.org/wiki/QR_algorithm>